# 05 · Files & Exceptions

So far, everything your programs knew disappeared when they stopped. In this notebook you learn two things that make programs **real**:

1. **Files**: save data so it is still there tomorrow (text files and JSON).
2. **Exceptions**: handle problems (a missing file, bad input, dividing by zero) without crashing.

They belong together: files are exactly where unexpected things happen.

**What's inside**

| # | Topic |
|---|---|
| 1 | Writing and reading text files |
| 2 | The `with` statement |
| 3 | File modes, and checking if a file exists |
| 4 | Saving structured data with JSON |
| 5 | What is an exception? |
| 6 | `try` / `except` |
| 7 | Several `except` blocks, `as e` |
| 8 | `else` and `finally` |
| 9 | Raising your own exceptions (`raise`) |
| 10 | Files + exceptions together: load and save |
| 11 | Practice problems |

> 🧹 This notebook creates a few small files in the same folder (like `notes.txt`). The last code cell deletes them.

---
## 1. Writing and reading text files

`open(filename, mode)` connects your program to a file. The **mode** says what you want to do:

| Mode | Meaning | If the file exists… |
|---|---|---|
| `"r"` | **read** (default) | must exist, or you get an error |
| `"w"` | **write** | its old content is **erased** ⚠️ |
| `"a"` | **append** (add to the end) | content is kept |

After working with a file you must **close** it.

### 📚 From class

In [1]:
file = open("notes.txt", "w")
file.write("Hello, this is my first file!\n")     # \n = new line
file.write("This is a note.\n")
file.close()

Look in your folder: a new file `notes.txt` appeared. Now read it back:

In [2]:
file = open("notes.txt", "r")
content = file.read()
file.close()
print(content)

Hello, this is my first file!
This is a note.



### 🆕 More examples

`write()` does **not** add a new line by itself. Forget `\n` and everything is glued together:

In [3]:
f = open("glued.txt", "w")
f.write("one")
f.write("two")
f.write("three")
f.close()

print(open("glued.txt").read())

onetwothree


---
## 2. The `with` statement

If your program crashes before `file.close()`, the file may stay open and data can be lost. The `with` statement closes the file **automatically**, even if there is an error. **Always use `with`.**

```
with open("file.txt", "r") as file:
    ...work with the file...
# the file is closed here
```

### 📚 From class

In [4]:
with open("notes.txt", "r") as file:       # with is used to open the file safely
    content = file.read(5)                  # read only the first 5 characters
    print(content)

Hello


In [5]:
titles = ["Dune\n", "The Hobbit\n", "1984\n", "To Kill a Mockingbird\n"]
with open("library.txt", "w") as file:
    file.writelines(titles)                 # write a whole list of lines

### Ways to read

| Method | Gives you |
|---|---|
| `file.read()` | the whole file as **one string** |
| `file.read(n)` | the first `n` characters |
| `file.readline()` | **one line** (including its `\n`) |
| `file.readlines()` | a **list** of all lines |
| `for line in file:` | one line at a time (best for big files) |

In [6]:
with open("library.txt", "r") as file:
    print(file.readline())        # first line only
    print(file.readline())        # the next line

Dune

The Hobbit



In [7]:
with open("library.txt", "r") as file:
    lines = file.readlines()
print(lines)

['Dune\n', 'The Hobbit\n', '1984\n', 'To Kill a Mockingbird\n']


In [8]:
with open("library.txt", "r") as file:
    for line in file:
        print(line.strip())          # strip() removes the \n at the end of each line

Dune
The Hobbit
1984
To Kill a Mockingbird


### 🆕 More examples

**Count the lines and words** in a file:

In [9]:
with open("library.txt", "r") as file:
    text = file.read()

print("Lines:", len(text.splitlines()))
print("Words:", len(text.split()))
print("Characters:", len(text))

Lines: 4
Words: 8
Characters: 43


**Write numbered lines** with a loop:

In [10]:
with open("squares.txt", "w") as file:
    for n in range(1, 6):
        file.write(f"{n} squared is {n ** 2}\n")

with open("squares.txt") as file:           # "r" is the default mode
    print(file.read())

1 squared is 1
2 squared is 4
3 squared is 9
4 squared is 16
5 squared is 25



---
## 3. File modes, and checking if a file exists

### Append mode
`"w"` **erases** the old content. To keep it and add more, use `"a"`.

In [11]:
with open("library.txt", "a") as file:
    file.write("Harry Potter\n")

with open("library.txt", "r") as file:
    print(file.read())

Dune
The Hobbit
1984
To Kill a Mockingbird
Harry Potter



### Does the file exist?
Reading a missing file is an error (`FileNotFoundError`). The `os` module can check first, and also delete files:

In [12]:
import os

if os.path.exists("library.txt"):
    print("File found!")
else:
    print("File does not exist yet.")

os.remove("squares.txt")                       # delete a file
print(os.path.exists("squares.txt"))

File found!
False


In [13]:
with open("no_such_file.txt", "r") as file:
    print(file.read())

FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.txt'

That error is the perfect lead-in for the second half of this notebook.

> ⚠️ **Common mistakes:** (1) opening with `"w"` and wiping a file you wanted to keep; (2) forgetting that `read()` gives a **string**, so numbers inside need `int(...)`; (3) forgetting `\n` when writing lines.

---
## 4. Saving structured data with JSON

Text files only store **text**. If you save a list of dictionaries with `str(...)`, it comes back as one long string, not a list!

In [14]:
books = [{"title": "Dune", "author": "Frank Herbert", "available": True}]

with open("books_text.txt", "w") as file:
    file.write(str(books))

with open("books_text.txt", "r") as file:
    content = file.read()

print(type(content))         # <class 'str'>: NOT a list anymore!
print(content[0])            # just the first character: "["

<class 'str'>
[


**JSON** (JavaScript Object Notation) is a standard text format for saving lists, dictionaries, strings, numbers and booleans. Almost every programming language and website understands it. The `json` module converts in both directions:

| Function | Direction |
|---|---|
| `json.dump(data, file)` | Python object → **writes** JSON to a file |
| `json.load(file)` | **reads** JSON from a file → Python object |
| `json.dumps(data)` / `json.loads(text)` | same, but to/from a **string** |

`indent=4` makes the file easy for humans to read.

### 📚 From class

In [15]:
import json

books = [{"title": "Dune", "author": "Frank Herbert", "available": True}]

with open("library.json", "w") as file:
    json.dump(books, file, indent=4)

In [16]:
with open("library.json", "r") as file:
    books = json.load(file)

print(type(books))             # <class 'list'>: a REAL list again!
print(books[0]["title"])

<class 'list'>
Dune


Let's peek at what the file looks like:

In [17]:
print(open("library.json").read())

[
    {
        "title": "Dune",
        "author": "Frank Herbert",
        "available": true
    }
]


### 🆕 More examples

Python ↔ JSON translation: `True` becomes `true`, `None` becomes `null`, and **tuples become lists**, because JSON has no tuple.

In [18]:
data = {"name": "Sara", "scores": [90, 85], "passed": True, "nickname": None, "point": (3, 4)}
text = json.dumps(data)
print(text)

back = json.loads(text)
print(back)
print(back["point"], type(back["point"]))     # a list now

{"name": "Sara", "scores": [90, 85], "passed": true, "nickname": null, "point": [3, 4]}
{'name': 'Sara', 'scores': [90, 85], 'passed': True, 'nickname': None, 'point': [3, 4]}
[3, 4] <class 'list'>


**Typical flow of a program that remembers things:** load at the start → change data while running → save after each change.

In [19]:
import os

def load_scores(filename):
    if os.path.exists(filename):
        with open(filename, "r") as file:
            return json.load(file)
    return {}

def save_scores(filename, scores):
    with open(filename, "w") as file:
        json.dump(scores, file, indent=4)

scores = load_scores("scores.json")           # first run: nothing there, so {}
scores["Ali"] = scores.get("Ali", 0) + 10
save_scores("scores.json", scores)

print(load_scores("scores.json"))

{'Ali': 10}


---
## 5. What is an exception?

When Python can't do what you asked, it **raises an exception**. If nothing handles it, the program stops with a traceback (you've seen plenty of these). Common ones:

| Exception | Happens when |
|---|---|
| `ValueError` | right type, wrong value: `int("abc")` |
| `TypeError` | wrong type: `"a" + 1` |
| `ZeroDivisionError` | `10 / 0` |
| `FileNotFoundError` | opening a file that isn't there |
| `IndexError` | list position doesn't exist |
| `KeyError` | dictionary key doesn't exist |
| `NameError` | variable was never defined |

**The idea of exception handling:** *"Try this. If something goes wrong, do that instead of crashing."*

### 📚 From class

In [20]:
age = int(input("Enter your age: "))
print("Next year you'll be", age + 1)

Enter your age: abc


ValueError: invalid literal for int() with base 10: 'abc'

---
## 6. `try` / `except`

```
try:
    risky code
except SomeError:
    what to do if that error happens
```

If the `try` block finishes without a problem, the `except` block is skipped. If an error happens, Python **jumps** straight to the matching `except`.

In [21]:
try:
    age = int(input("Enter your age: "))
    print("Next year you'll be", age + 1)
except:
    print("That wasn't a valid number.")

Enter your age: abc
That wasn't a valid number.


That works, but a bare `except:` catches **every** problem, even ones you didn't expect, which hides bugs. Name the error you expect:

In [22]:
try:
    age = int(input("Enter your age: "))
except ValueError:
    print("Please enter a valid whole number.")

Enter your age: twenty
Please enter a valid whole number.


### 🆕 More examples

**Keep asking until the input is valid**: a `while True` loop with `try` inside and `break` on success. You will use this pattern again and again.

In [23]:
while True:
    try:
        age = int(input("Enter your age: "))
        break                          # success: leave the loop
    except ValueError:
        print("Numbers only, please!")

print("Thanks! You are", age)

Enter your age: abc
Numbers only, please!
Enter your age: 12.5
Numbers only, please!
Enter your age: 15
Thanks! You are 15


---
## 7. Several `except` blocks, `as e`

Different problems need different responses. Add several `except` blocks. Python uses the **first** one that matches.

### 📚 From class

In [24]:
try:
    num1 = int(input("Enter a number: "))
    num2 = int(input("Enter another number: "))
    print(num1 / num2)
except ValueError:
    print("Please enter valid numbers.")
except ZeroDivisionError:
    print("You can't divide by zero!")
except Exception:
    print("Something unexpected happened.")

Enter a number: 10
Enter another number: 0
You can't divide by zero!


In [25]:
try:
    num1 = int(input("Enter a number: "))
    num2 = int(input("Enter another number: "))
    print(num1 / num2)
except ValueError:
    print("Please enter valid numbers.")
except ZeroDivisionError:
    print("You can't divide by zero!")

Enter a number: ten
Please enter valid numbers.


> 💡 `Exception` is the **parent** of almost all errors, so it goes **last** as a safety net. If you put it first, it would catch everything and the specific blocks would never run.

### Getting the error message: `as e`
Python can hand you the exception object so you can show or log the original message:

In [26]:
try:
    numbers = [1, 2, 3]
    print(numbers[10])
except IndexError as e:
    print("Oops:", e)
    print("Type of error:", type(e).__name__)

Oops: list index out of range
Type of error: IndexError


### 🆕 More examples

Catch **several error types in one block** with a tuple:

In [27]:
for value in ["42", "abc", None]:
    try:
        print(int(value) * 2)
    except (ValueError, TypeError) as e:
        print(f"Can't convert {value!r}: {e}")

84
Can't convert 'abc': invalid literal for int() with base 10: 'abc'
Can't convert None: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


Safe dictionary lookup: `KeyError`, or use `.get`:

In [28]:
prices = {"apple": 120}
try:
    print(prices["mango"])
except KeyError:
    print("We don't sell mango.")

print(prices.get("mango", "We don't sell mango."))    # the same, with no try needed

We don't sell mango.
We don't sell mango.


---
## 8. `else` and `finally`

| Block | Runs when… |
|---|---|
| `try` | always first |
| `except` | an error happened |
| `else` | **no** error happened |
| `finally` | **always**, error or not (clean-up: closing files, saying goodbye) |

### 📚 From class

In [29]:
try:
    num = int(input("Enter a number: "))
except ValueError:
    print("That wasn't a number.")
else:
    print("Great, you entered:", num)
finally:
    print("Thanks for trying!")

Enter a number: 42
Great, you entered: 42
Thanks for trying!


In [30]:
try:
    num = int(input("Enter a number: "))
except ValueError:
    print("That wasn't a number.")
else:
    print("Great, you entered:", num)
finally:
    print("Thanks for trying!")

Enter a number: oops
That wasn't a number.
Thanks for trying!


### 🆕 More examples

`finally` is perfect for clean-up that must happen no matter what:

In [31]:
def risky_division(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
    finally:
        print(f"(finished dividing {a} by {b})")

print(risky_division(10, 2))
print(risky_division(10, 0))

(finished dividing 10 by 2)
5.0
(finished dividing 10 by 0)
None


> 💡 Notice that `finally` runs even though the function already hit a `return`.

---
## 9. Raising your own exceptions (`raise`)

You can **raise** an exception yourself when your code detects a bad situation. This is how a function says *"I can't do that"*. The caller decides how to react.

### 📚 From class

In [32]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("Insufficient balance!")
    return balance - amount

try:
    balance = withdraw(100, 500)
except ValueError as e:
    print("Error:", e)

Error: Insufficient balance!


### 🆕 More examples

In [33]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("Age must be a whole number.")
    if age < 0 or age > 120:
        raise ValueError("Age must be between 0 and 120.")
    return age

for test in [25, -5, "ten"]:
    try:
        print("OK:", set_age(test))
    except (TypeError, ValueError) as e:
        print("Rejected:", e)

OK: 25
Rejected: Age must be between 0 and 120.
Rejected: Age must be a whole number.


### Custom exceptions *(bonus)*
You can create your own exception type by making a class that inherits from `Exception`. This makes your errors very descriptive. (You will use this in the bank project after learning classes.)

In [34]:
class InsufficientFundsError(Exception):
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"Need {amount - balance} more.")
    return balance - amount

try:
    withdraw(100, 150)
except InsufficientFundsError as e:
    print("Transaction declined:", e)

Transaction declined: Need 50 more.


---
## 10. Files + exceptions together: load and save

Back to the problem from the start: what if the file doesn't exist yet, or got damaged? A robust `load` function **expects** those problems.

### 📚 From class

In [35]:
import json

def load_books():
    try:
        with open("lib_books.json", "r") as file:
            return json.load(file)
    except FileNotFoundError:
        return []                                  # first run: start with an empty list
    except json.JSONDecodeError:
        print("Warning: the file was corrupted. Starting fresh.")
        return []

print(load_books())

[]


Now simulate a **corrupted** file (invalid JSON) and see the second `except` in action:

In [36]:
with open("lib_books.json", "w") as file:
    file.write("this is not valid json {{{")

print(load_books())

[]


### 🆕 More examples

**A complete mini app:** a note-taker that remembers notes between runs. It combines loops, `match`, functions, JSON, and exceptions.

In [37]:
import json

FILENAME = "my_notes.json"

def load_notes():
    try:
        with open(FILENAME) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_notes(notes):
    with open(FILENAME, "w") as f:
        json.dump(notes, f, indent=2)

notes = load_notes()
print(f"Loaded {len(notes)} saved notes.")

while True:
    print("\n1. Add note   2. Show notes   3. Quit")
    match input("Choose: "):
        case "1":
            notes.append(input("Write your note: "))
            save_notes(notes)
            print("Saved!")
        case "2":
            for i, note in enumerate(notes, start=1):
                print(f"{i}. {note}")
        case "3":
            break
        case _:
            print("Invalid choice")

Loaded 0 saved notes.

1. Add note   2. Show notes   3. Quit
Choose: 1
Write your note: Buy milk
Saved!

1. Add note   2. Show notes   3. Quit
Choose: 1
Write your note: Finish homework
Saved!

1. Add note   2. Show notes   3. Quit
Choose: 2
1. Buy milk
2. Finish homework

1. Add note   2. Show notes   3. Quit
Choose: 3


Run that cell **again**: it reports "Loaded 2 saved notes" because the notes were saved to disk. (When you run it yourself, Jupyter will wait for you to type.)

### Cleaning up the files we made

In [38]:
import os
for name in ["notes.txt", "glued.txt", "library.txt", "books_text.txt", "library.json",
             "scores.json", "lib_books.json", "my_notes.json"]:
    if os.path.exists(name):
        os.remove(name)
        print("deleted", name)

deleted notes.txt
deleted glued.txt
deleted library.txt
deleted books_text.txt
deleted library.json
deleted scores.json
deleted lib_books.json
deleted my_notes.json


> 💡 **Style note:** Python programmers prefer *"easier to ask forgiveness than permission"*: just `try` the action and handle the failure, rather than checking everything first.

---
## 11. Practice problems

**🟢 Easy**
1. Write three lines about yourself into `about_me.txt`, then read and print them.
2. Append a fourth line without erasing the first three.
3. Ask for a number and print `100 / number`, handling both a non-number and zero.

**🟡 Medium**
4. **Safe input function:** write `ask_int(prompt)` that keeps asking until the user types a valid integer, then returns it.
5. Count how many times each word appears in a text file and save the result as JSON.
6. **To-do app:** like the note-taker above, but with a "mark done" option, saved in JSON as a list of dictionaries (`{"task": ..., "done": ...}`).

**🔴 Challenge**
7. A **contact book** saved in JSON with add / search / delete, that survives a missing or corrupted file.
8. Write a function `read_numbers(filename)` that reads one number per line, **skips** bad lines (printing a warning), and returns the list of valid numbers.

In [39]:
# your solutions here

---
### ✅ You can now…
- write, read and append text files safely with `with`
- save and load lists and dictionaries with JSON
- read an exception name and respond to it with `try` / `except`
- use `else`, `finally`, `raise`, and `as e`
- build programs that remember data and survive bad input

**Next notebook →** `06_OOP_Part1_Classes_and_Objects.ipynb`